# Chapter 15 - Affine Quadrics

Source span: printed Volume II pages 146-169; PDF pages 155-178.

An affine quadric is the first nonlinear object that still behaves like linear algebra after the right change of coordinates. Lines and planes have one equation of degree one. Affine quadrics have one equation of degree two, so their geometry is governed by a symmetric matrix, a linear term, and a constant term:

$$
q(x)=x^T A x + 2 b^T x + c, \qquad x\in K^n.
$$

The chapter question is: how much of a curve or surface can be understood by reducing this expression, and what remains visible after we forget projective coordinates and work in an affine chart? The answer has two parts. Algebraically, the symmetric part of the quadratic term controls directions, rank, and signature. Affinely, the linear term decides whether the object has a center or whether it opens like a parabola. Geometrically, polarity and Euclidean normal forms turn these computations into centers, diameters, axes, tangent planes, and familiar models such as ellipsoids, hyperboloids, paraboloids, cylinders, and cones.

This notebook is a standalone computational tour. It avoids treating classification as a memorized table: each visual is paired with a reduction, residual check, or invariant that can be recomputed.


## Translation guide

| Book idea | Computational translation |
| --- | --- |
| Affine quadric in $K^n$ | Zero set of `q(x) = x.T @ A @ x + 2*b.T @ x + c` with `A` symmetric. |
| Proper affine quadric | A nontrivial zero set whose homogenized projective quadric is not just the hyperplane at infinity. |
| Change of affine coordinates | Replace `x` by `M @ y + p`; congruence acts on `A`, while `p` changes the linear and constant terms. |
| Center | A solution of `A @ x + b = 0`; when `A` is invertible the center is unique. |
| Diameter direction | Parallel chords have midpoints in an affine hyperplane; in centered coordinates this is controlled by the bilinear form `u.T @ A @ v`. |
| Polarity | Homogenize the equation and use the symmetric matrix `C`; the polar hyperplane of `p` is `C @ p`. |
| Euclidean axes | Diagonalize a real symmetric `A` with an orthogonal matrix; the eigenvectors are principal directions. |
| Topological type | Read from normal form, rank, signature, and whether the level set is empty, compact, connected, or split into sheets. |

Two warnings keep the computations honest. First, a projective quadric can meet the hyperplane at infinity, so the affine piece may have missing directions or asymptotes. Second, diagonalizing the quadratic term alone is not the whole affine problem: after diagonalization, remaining linear terms along null directions are exactly where parabolic behavior appears.


## Visualization storyboard

1. **Affine reduction panels.** Contours show how completing squares moves a conic to its center when a center exists, and why a parabolic direction survives when the quadratic part has a null direction.
2. **Normal form table.** Rank, signature, and residual linear terms become an executable classification checklist for real affine quadrics.
3. **Interactive surface gallery.** Six standard three-dimensional models are placed side by side so the eye can connect signs in the equation to compactness, connectedness, and ends.
4. **Topology slices.** Cross-sections isolate the qualitative difference between ellipsoid, one-sheet and two-sheet hyperboloids, cones, and paraboloids.
5. **Polarity and diameters.** A conic diagram computes a polar line, tangent contact points, and conjugate diameter directions from the same symmetric matrix.
6. **Euclidean axes lab.** A rotated translated ellipsoid is reduced numerically; the computed center, axes, and residuals become the final sanity checks.


In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def discover_book_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "utils").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Geometry II book root")


BOOK_ROOT = discover_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, assert_artifacts, display_artifact, save_csv, save_json, save_matplotlib, save_plotly_html
from utils.geometry import conic_residual, homogeneous, polar_line, quadratic_signature
from utils.plotting import COLORS, save_figure, set_equal_3d

CHAPTER = 15
ARTIFACT_KINDS = ("figures", "plots", "tables", "checks")
for kind in ARTIFACT_KINDS:
    (BOOK_ROOT / "artifacts" / f"chapter-{CHAPTER:02d}" / kind).mkdir(parents=True, exist_ok=True)


def chapter_artifact(kind, filename):
    return artifact_path(CHAPTER, kind, filename, book_root=BOOK_ROOT)


np.set_printoptions(precision=4, suppress=True)
BOOK_ROOT.name


## 1. From an equation to an affine object

The expression `q(x)` has three layers. The matrix `A` is the quadratic part, the vector `b` is the linear part, and `c` fixes the level. Symmetrizing `A` changes nothing because only the symmetric part contributes to `x^T A x`.

A translation by `x0` changes the equation to

$$
q(x_0+y)=y^T A y + 2(Ax_0+b)^T y + q(x_0).
$$

So a center is exactly a point where the gradient's affine part vanishes: `A @ x0 + b = 0`. If `A` is nonsingular, this equation has a unique solution and every centered chord is paired with an opposite chord. If `A` has a null direction, the linear term may survive along that direction; this is the computational source of affine paraboloids.

The next cell builds small reduction functions and checks this identity numerically.


In [ ]:
def symmetrize(A):
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def quadric_value(A, b, c, points):
    A = symmetrize(A)
    b = np.asarray(b, dtype=float)
    X = np.asarray(points, dtype=float)
    return np.einsum("...i,ij,...j->...", X, A, X) + 2 * (X @ b) + c


def centered_reduction(A, b, c):
    A = symmetrize(A)
    b = np.asarray(b, dtype=float)
    center = -np.linalg.solve(A, b)
    constant = float(quadric_value(A, b, c, center))
    eigvals, eigvecs = np.linalg.eigh(A)
    return {"center": center, "constant": constant, "eigenvalues": eigvals, "eigenvectors": eigvecs}


A_demo = np.array([[2.0, 0.45], [0.45, 1.15]])
b_demo = np.array([-1.1, 0.55])
c_demo = -1.0
reduction_demo = centered_reduction(A_demo, b_demo, c_demo)

rng = np.random.default_rng(1515)
y_samples = rng.normal(size=(8, 2))
left = quadric_value(A_demo, b_demo, c_demo, reduction_demo["center"] + y_samples)
right = np.einsum("...i,ij,...j->...", y_samples, A_demo, y_samples) + reduction_demo["constant"]
translation_error = float(np.max(np.abs(left - right)))

translation_check = {
    "center": reduction_demo["center"].round(6).tolist(),
    "eigenvalues": reduction_demo["eigenvalues"].round(6).tolist(),
    "constant_at_center": round(reduction_demo["constant"], 6),
    "max_translation_identity_error": translation_error,
}
translation_check


## 2. Completing squares as a picture

The first panel below shows a full-rank quadratic part: translating to the center removes the linear term and turns the equation into a pure quadratic level set. The second panel shows the same principle for a hyperbola. The third panel deliberately has a rank-one quadratic part. There is no unique center; after diagonalizing the quadratic direction, a linear term remains in the null direction, producing a parabola. The fourth panel is the limiting cylindrical behavior: one direction is unconstrained because neither a quadratic nor a linear term uses it.

This is the affine version of the usual projective story. Projective transformations can move pieces through infinity; affine transformations preserve the hyperplane at infinity, so the directions where `A` degenerates become visible.


In [ ]:
def contour_panel(ax, A, b, c, xlim, ylim, title, *, color=COLORS["blue"], center=None):
    xs = np.linspace(xlim[0], xlim[1], 420)
    ys = np.linspace(ylim[0], ylim[1], 420)
    X, Y = np.meshgrid(xs, ys)
    pts = np.column_stack([X.ravel(), Y.ravel()])
    Z = quadric_value(A, b, c, pts).reshape(X.shape)
    ax.contour(X, Y, Z, levels=[0.0], colors=[color], linewidths=2.5)
    ax.contour(X, Y, Z, levels=[-1.0, 1.0], colors=["#d1d5db"], linewidths=0.9, linestyles="dashed")
    ax.axhline(0, color="#d1d5db", linewidth=0.8)
    ax.axvline(0, color="#d1d5db", linewidth=0.8)
    if center is not None:
        ax.scatter([center[0]], [center[1]], s=48, color=COLORS["orange"], edgecolor="white", zorder=5)
        ax.annotate("center", center + np.array([0.08, 0.08]), fontsize=8, color=COLORS["orange"])
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#eef0f3", linewidth=0.7)
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")


fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.2))

A1 = np.array([[2.0, 0.35], [0.35, 1.0]])
b1 = np.array([-1.0, 0.4])
c1 = -1.1
center1 = centered_reduction(A1, b1, c1)["center"]
contour_panel(axes[0, 0], A1, b1, c1, (-2.2, 2.2), (-2.0, 2.0), "full rank: translate to a center", center=center1)

A2 = np.array([[1.0, 0.25], [0.25, -0.75]])
b2 = np.array([0.35, -0.2])
c2 = -0.2
center2 = centered_reduction(A2, b2, c2)["center"]
contour_panel(axes[0, 1], A2, b2, c2, (-2.7, 2.7), (-2.4, 2.4), "indefinite full rank: asymptotic directions", color=COLORS["teal"], center=center2)

A3_rank = np.array([[1.0, 0.0], [0.0, 0.0]])
b3_rank = np.array([0.0, -0.5])
c3_rank = 0.0
contour_panel(axes[1, 0], A3_rank, b3_rank, c3_rank, (-2.4, 2.4), (-0.6, 5.2), "rank one with surviving linear term", color=COLORS["purple"])
axes[1, 0].annotate("null direction carries the opening", xy=(1.15, 1.35), xytext=(-2.1, 4.25), arrowprops={"arrowstyle": "->", "color": COLORS["gray"]}, fontsize=8)

A4 = np.array([[1.0, 0.0], [0.0, 0.0]])
b4 = np.array([0.0, 0.0])
c4 = -1.0
contour_panel(axes[1, 1], A4, b4, c4, (-2.4, 2.4), (-2.4, 2.4), "rank one with free affine direction", color=COLORS["red"])
axes[1, 1].annotate("cylindrical direction", xy=(1.0, 1.3), xytext=(1.2, -1.8), arrowprops={"arrowstyle": "->", "color": COLORS["gray"]}, fontsize=8)

for ax in axes.ravel():
    ax.tick_params(labelsize=8)
fig.suptitle("Affine reduction: center, parabola, cylinder", x=0.02, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
reduction_path = chapter_artifact("figures", "affine_reduction_panels.png")
save_matplotlib(fig, reduction_path)
plt.close(fig)
display_artifact(reduction_path, width=900)


## 3. Classification as executable data

Over the real numbers, the signs of the diagonalized quadratic part matter. Over the complex numbers, nonzero coefficients can be rescaled, so rank and the leftover affine linear behavior carry more of the classification burden. In either setting, the practical procedure is the same:

1. Symmetrize `A`.
2. Diagonalize or otherwise reduce the quadratic part by congruence.
3. Translate away every linear term that lies in the image of `A`.
4. Inspect the remaining linear terms along the nullspace.
5. Normalize the constant and nonzero coefficients.

The table is not meant to be exhaustive in every dimension. It is a working checklist for recognizing the models that dominate affine geometry in dimensions two and three.


In [ ]:
normal_forms = [
    {"model": "ellipse / ellipsoid", "prototype": "x1^2/a1^2 + ... + xr^2/ar^2 = 1", "rank_A": "r", "signature": "all one sign", "center": "yes", "real_topology": "compact sphere when r=n"},
    {"model": "imaginary ellipsoid", "prototype": "x1^2/a1^2 + ... + xr^2/ar^2 = -1", "rank_A": "r", "signature": "all one sign", "center": "yes", "real_topology": "empty over R"},
    {"model": "one-sheet hyperboloid", "prototype": "x^2/a^2 + y^2/b^2 - z^2/c^2 = 1", "rank_A": "3", "signature": "two plus, one minus", "center": "yes", "real_topology": "connected, S^1 times R"},
    {"model": "two-sheet hyperboloid", "prototype": "z^2/c^2 - x^2/a^2 - y^2/b^2 = 1", "rank_A": "3", "signature": "one plus, two minus", "center": "yes", "real_topology": "two components"},
    {"model": "quadratic cone", "prototype": "x^2/a^2 + y^2/b^2 - z^2/c^2 = 0", "rank_A": "3", "signature": "mixed", "center": "singular", "real_topology": "two nappes meeting at vertex"},
    {"model": "elliptic paraboloid", "prototype": "z = x^2/a^2 + y^2/b^2", "rank_A": "2", "signature": "same sign on image", "center": "no", "real_topology": "R^2"},
    {"model": "hyperbolic paraboloid", "prototype": "z = x^2/a^2 - y^2/b^2", "rank_A": "2", "signature": "mixed on image", "center": "no", "real_topology": "R^2 with two rulings"},
    {"model": "cylinder", "prototype": "x^2/a^2 +/- y^2/b^2 = 1", "rank_A": "2", "signature": "depends on base", "center": "affine line of centers", "real_topology": "base conic times R"},
]
normal_forms_path = chapter_artifact("tables", "affine_normal_forms.csv")
save_csv(normal_forms, normal_forms_path)
pd.DataFrame(normal_forms)


## 4. Surface gallery: signs become shapes

A three-dimensional affine quadric can often be recognized before any formula is memorized. Positive signs enclosing a positive level make compact ovals. Mixed signs create escape directions. A zero level of a mixed form creates a cone. A missing quadratic direction, if still hit by a linear term, creates a paraboloid rather than a centered surface.

The gallery below is an HTML artifact. Rotate the surfaces when reading interactively: the purpose is to make the normal forms feel like one family rather than six isolated examples.


In [ ]:
def surface_ellipsoid(a=1.55, b=1.0, c=0.75, n=54):
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(0, np.pi, n)
    U, V = np.meshgrid(u, v)
    return a * np.cos(U) * np.sin(V), b * np.sin(U) * np.sin(V), c * np.cos(V)


def surface_one_sheet(a=1.0, b=0.75, c=0.9, n=58):
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(-1.25, 1.25, n)
    U, V = np.meshgrid(u, v)
    return a * np.cosh(V) * np.cos(U), b * np.cosh(V) * np.sin(U), c * np.sinh(V)


def surface_two_sheet(a=0.75, b=0.65, c=0.9, n=50):
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(0.0, 1.25, n)
    U, V = np.meshgrid(u, v)
    x = a * np.sinh(V) * np.cos(U)
    y = b * np.sinh(V) * np.sin(U)
    z = c * np.cosh(V)
    return [(x, y, z), (x, y, -z)]


def surface_paraboloid(kind="elliptic", n=56):
    x = np.linspace(-1.6, 1.6, n)
    y = np.linspace(-1.6, 1.6, n)
    X, Y = np.meshgrid(x, y)
    if kind == "elliptic":
        Z = 0.45 * X**2 + 0.8 * Y**2
    else:
        Z = 0.55 * X**2 - 0.75 * Y**2
    return X, Y, Z


def surface_cone(n=56):
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(-1.45, 1.45, n)
    U, V = np.meshgrid(u, v)
    R = np.abs(V)
    return R * np.cos(U), R * np.sin(U), V


fig = make_subplots(
    rows=2,
    cols=3,
    specs=[[{"type": "surface"}, {"type": "surface"}, {"type": "surface"}], [{"type": "surface"}, {"type": "surface"}, {"type": "surface"}]],
    subplot_titles=("ellipsoid", "one-sheet hyperboloid", "two-sheet hyperboloid", "elliptic paraboloid", "hyperbolic paraboloid", "quadratic cone"),
    horizontal_spacing=0.03,
    vertical_spacing=0.08,
)

colors = ["Viridis", "Teal", "Blues", "Greens", "RdBu", "Inferno"]
for idx, data in enumerate([surface_ellipsoid(), surface_one_sheet()], start=1):
    row, col = divmod(idx - 1, 3)
    fig.add_trace(go.Surface(x=data[0], y=data[1], z=data[2], colorscale=colors[idx - 1], showscale=False, opacity=0.92), row=row + 1, col=col + 1)

for data in surface_two_sheet():
    fig.add_trace(go.Surface(x=data[0], y=data[1], z=data[2], colorscale="Blues", showscale=False, opacity=0.92), row=1, col=3)

for idx, data in enumerate([surface_paraboloid("elliptic"), surface_paraboloid("hyperbolic"), surface_cone()], start=4):
    row, col = divmod(idx - 1, 3)
    fig.add_trace(go.Surface(x=data[0], y=data[1], z=data[2], colorscale=colors[idx - 1], showscale=False, opacity=0.92), row=row + 1, col=col + 1)

for scene_name in ["scene", "scene2", "scene3", "scene4", "scene5", "scene6"]:
    fig.update_layout(**{scene_name: {"xaxis": {"visible": False}, "yaxis": {"visible": False}, "zaxis": {"visible": False}, "aspectmode": "data"}})
fig.update_layout(title="Affine quadric normal forms in dimension three", margin={"l": 0, "r": 0, "t": 55, "b": 0}, height=760, showlegend=False)

gallery_path = chapter_artifact("plots", "quadric_classification_gallery.html")
save_plotly_html(fig, gallery_path)
display_artifact(gallery_path, width=980, height=780)


## 5. Topology from normal form

The topology of a real affine quadric is not an extra decoration after classification; it is one of the most useful things classification tells us. In a centered nondegenerate equation, the signs decide whether the zero set is compact, connected but open, or split into two pieces. In a parabolic equation, the missing quadratic direction becomes an unbounded height direction, yet the surface can still be connected and smooth.

The following cross-section view compresses the three-dimensional story into plane slices. A vertical slice through an ellipsoid is an oval. A slice through a one-sheet hyperboloid is a hyperbola whose branches are connected by the omitted circular direction. A two-sheet hyperboloid keeps the branches as separate components. A cone advertises a singular center. A paraboloid has no center at all.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14.5, 3.2))
xs = np.linspace(-2.4, 2.4, 600)

z = np.sqrt(np.maximum(0, 1 - xs**2 / 2.25))
mask = np.isfinite(z) & (z > 0)
axes[0].plot(xs[mask], z[mask], color=COLORS["blue"], linewidth=2.2)
axes[0].plot(xs[mask], -z[mask], color=COLORS["blue"], linewidth=2.2)
axes[0].set_title("compact oval", fontsize=10, fontweight="bold")

z = np.sqrt(np.maximum(0, xs**2 - 1))
mask = np.abs(xs) >= 1
axes[1].plot(xs[mask], z[mask], color=COLORS["teal"], linewidth=2.2)
axes[1].plot(xs[mask], -z[mask], color=COLORS["teal"], linewidth=2.2)
axes[1].set_title("one-sheet trace", fontsize=10, fontweight="bold")

z = np.sqrt(1 + xs**2)
axes[2].plot(xs, z, color=COLORS["orange"], linewidth=2.2)
axes[2].plot(xs, -z, color=COLORS["orange"], linewidth=2.2)
axes[2].set_title("two components", fontsize=10, fontweight="bold")

axes[3].plot(xs, np.abs(xs), color=COLORS["red"], linewidth=2.2)
axes[3].plot(xs, -np.abs(xs), color=COLORS["red"], linewidth=2.2)
axes[3].scatter([0], [0], color=COLORS["red"], s=32, zorder=5)
axes[3].set_title("singular cone", fontsize=10, fontweight="bold")

z = xs**2 / 1.25
axes[4].plot(xs, z, color=COLORS["purple"], linewidth=2.2)
axes[4].set_title("parabolic end", fontsize=10, fontweight="bold")

for ax in axes:
    ax.axhline(0, color="#d1d5db", linewidth=0.8)
    ax.axvline(0, color="#d1d5db", linewidth=0.8)
    ax.set_xlim(-2.3, 2.3)
    ax.set_ylim(-2.2, 2.8)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#eef0f3", linewidth=0.7)
    ax.tick_params(labelsize=7)

fig.suptitle("Topology clues from two-dimensional slices", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.92))
topology_path = chapter_artifact("figures", "topology_slices.png")
save_figure(fig, topology_path)
display_artifact(topology_path, width=980)


## 6. Polarity, tangent planes, and diameters

Homogenizing an affine equation brings back the projective tool of polarity. For a conic written as

$$
X^T C X=0, \qquad X=(x,y,1),
$$

the polar line of a point `P` is `(C @ P)^T X = 0`. If `P` lies outside an ellipse, that line is the chord joining the two tangent contact points. If `P` is on the conic, the same formula gives the tangent line at `P`.

For a centered affine quadric, diameters are described by the bilinear form attached to `A`. Two directions `u` and `v` are conjugate when `u.T @ A @ v = 0`. This is the affine algebra behind the classical picture of paired diameters in conics and quadrics.


In [ ]:
def line_points(line, xlim=(-3.0, 3.4), ylim=(-2.0, 2.0), count=200):
    a, b, c = line
    xs = np.linspace(xlim[0], xlim[1], count)
    if abs(b) > 1e-12:
        ys = -(a * xs + c) / b
        pts = np.column_stack([xs, ys])
        mask = (ys >= ylim[0]) & (ys <= ylim[1])
        return pts[mask]
    x = -c / a
    ys = np.linspace(ylim[0], ylim[1], count)
    return np.column_stack([np.full_like(ys, x), ys])


def tangent_contact_parameters_for_ellipse(a_axis, b_axis, line):
    alpha = line[0] * a_axis
    beta = line[1] * b_axis
    gamma = line[2]
    radius = math.hypot(alpha, beta)
    phase = math.atan2(beta, alpha)
    if abs(gamma) > radius + 1e-10:
        return []
    delta = math.acos(np.clip(-gamma / radius, -1, 1))
    return [phase + delta, phase - delta]


C = np.diag([1 / 4, 1.0, -1.0])
A_ellipse = C[:2, :2]
external_point = np.array([3.0, 1.0, 1.0])
polar = polar_line(C, external_point)
contact_ts = tangent_contact_parameters_for_ellipse(2.0, 1.0, polar)
contacts = np.array([[2 * np.cos(t), np.sin(t)] for t in contact_ts])

u = np.array([1.0, 0.5])
v = np.array([2.0, -1.0])
conjugacy_residual = float(u @ A_ellipse @ v)

fig, ax = plt.subplots(figsize=(8.0, 5.8))
t = np.linspace(0, 2 * np.pi, 500)
ellipse = np.column_stack([2 * np.cos(t), np.sin(t)])
ax.plot(ellipse[:, 0], ellipse[:, 1], color=COLORS["blue"], linewidth=2.4, label="ellipse")

pp = line_points(polar, xlim=(-2.4, 3.4), ylim=(-1.7, 1.7))
ax.plot(pp[:, 0], pp[:, 1], color=COLORS["orange"], linewidth=2.0, label="polar line")
ax.scatter([external_point[0]], [external_point[1]], color=COLORS["red"], s=56, edgecolor="white", zorder=6)
ax.annotate("P", (external_point[0] + 0.08, external_point[1] + 0.05), color=COLORS["red"], fontsize=10)

for point in contacts:
    tangent = polar_line(C, np.array([point[0], point[1], 1.0]))
    tp = line_points(tangent, xlim=(-2.4, 3.4), ylim=(-1.7, 1.7))
    ax.plot(tp[:, 0], tp[:, 1], color="#9ca3af", linewidth=1.25, linestyle="--")
ax.scatter(contacts[:, 0], contacts[:, 1], color=COLORS["orange"], s=42, edgecolor="white", zorder=6, label="contact points")

for direction, label, color in [(u, "u", COLORS["teal"]), (v, "v", COLORS["purple"])]:
    direction = direction / np.linalg.norm(direction)
    segment = np.vstack([-2.1 * direction, 2.1 * direction])
    ax.plot(segment[:, 0], segment[:, 1], color=color, linewidth=2.0, alpha=0.85)
    ax.annotate(label, segment[-1] + np.array([0.05, 0.05]), color=color, fontsize=10)

ax.scatter([0], [0], color=COLORS["ink"], s=30, zorder=6)
ax.annotate("center", (0.08, -0.16), fontsize=8, color=COLORS["ink"])
ax.set_xlim(-2.45, 3.35)
ax.set_ylim(-1.65, 1.65)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, color="#eef0f3", linewidth=0.7)
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Polarity and conjugate diameters from one matrix", loc="left", fontsize=13, fontweight="bold")
polarity_path = chapter_artifact("figures", "polarity_and_diameters.png")
save_figure(fig, polarity_path)
display_artifact(polarity_path, width=860)

polarity_check = {
    "polar_line_coefficients": polar.round(8).tolist(),
    "contact_residuals": conic_residual(C, contacts).round(10).tolist(),
    "point_on_tangent_residuals": [float(np.dot(polar_line(C, np.r_[q, 1.0]), external_point)) for q in contacts],
    "conjugacy_residual": conjugacy_residual,
}
polarity_check


## 7. Euclidean normal forms and measured axes

Affine classification allows general invertible changes of coordinates. Euclidean classification is stricter: rotations and translations are allowed, but lengths and angles matter. For a real symmetric matrix `A`, the spectral theorem gives an orthonormal eigenbasis. After moving to the center, an ellipsoid has the form

$$
\lambda_1 y_1^2 + \lambda_2 y_2^2 + \lambda_3 y_3^2 = -q(x_0),
$$

so the semi-axis lengths are `sqrt(-q(x0) / lambda_i)` when all eigenvalues and `-q(x0)` are positive. This is why Euclidean quadrics carry numerical invariants, not just sign patterns.


In [ ]:
def rotation_matrix_zxy(alpha, beta, gamma):
    ca, sa = math.cos(alpha), math.sin(alpha)
    cb, sb = math.cos(beta), math.sin(beta)
    cg, sg = math.cos(gamma), math.sin(gamma)
    Rz = np.array([[ca, -sa, 0], [sa, ca, 0], [0, 0, 1]])
    Rx = np.array([[1, 0, 0], [0, cb, -sb], [0, sb, cb]])
    Ry = np.array([[cg, 0, sg], [0, 1, 0], [-sg, 0, cg]])
    return Rz @ Rx @ Ry


true_center = np.array([0.8, -0.55, 0.35])
true_axes = np.array([1.8, 1.05, 0.65])
R = rotation_matrix_zxy(0.45, -0.35, 0.25)
Lambda = np.diag(1 / true_axes**2)
A3 = R @ Lambda @ R.T
b3 = -A3 @ true_center
c3 = float(true_center @ A3 @ true_center - 1.0)

reduced = centered_reduction(A3, b3, c3)
computed_axes = np.sqrt(-reduced["constant"] / reduced["eigenvalues"])
axis_order = np.argsort(computed_axes)[::-1]
computed_axes_sorted = computed_axes[axis_order]
computed_directions = reduced["eigenvectors"][:, axis_order]

u_grid = np.linspace(0, 2 * np.pi, 44)
v_grid = np.linspace(0, np.pi, 28)
U, V = np.meshgrid(u_grid, v_grid)
local = np.stack([
    computed_axes_sorted[0] * np.cos(U) * np.sin(V),
    computed_axes_sorted[1] * np.sin(U) * np.sin(V),
    computed_axes_sorted[2] * np.cos(V),
], axis=-1)
world = local @ computed_directions.T + reduced["center"]

fig = plt.figure(figsize=(8.2, 6.4))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(world[..., 0], world[..., 1], world[..., 2], color="#8ecae6", edgecolor="#ffffff", linewidth=0.25, alpha=0.82)
colors_axes = [COLORS["red"], COLORS["teal"], COLORS["purple"]]
for length, direction, color, label in zip(computed_axes_sorted, computed_directions.T, colors_axes, ["a1", "a2", "a3"]):
    start = reduced["center"] - length * direction
    end = reduced["center"] + length * direction
    ax.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], color=color, linewidth=3.0)
    ax.text(end[0], end[1], end[2], label, color=color, fontsize=9)
ax.scatter([reduced["center"][0]], [reduced["center"][1]], [reduced["center"][2]], color=COLORS["ink"], s=32)
ax.set_title("Euclidean reduction: center and principal axes", loc="left", fontsize=13, fontweight="bold")
all_points = world.reshape(-1, 3)
set_equal_3d(ax, all_points, radius=2.25)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
axes_path = chapter_artifact("figures", "euclidean_axes_lab.png")
save_figure(fig, axes_path)
display_artifact(axes_path, width=820)

axis_lab_check = {
    "true_center": true_center.round(8).tolist(),
    "computed_center": reduced["center"].round(8).tolist(),
    "true_axes_sorted": sorted(true_axes.tolist(), reverse=True),
    "computed_axes_sorted": computed_axes_sorted.round(8).tolist(),
    "signature_A": quadratic_signature(A3),
    "center_error": float(np.linalg.norm(reduced["center"] - true_center)),
    "axis_length_error": float(np.linalg.norm(np.sort(computed_axes) - np.sort(true_axes))),
}
axis_lab_check


## 8. Applied lab: classify a quadric from samples

Suppose a dataset claims to come from a translated ellipsoid, but the coordinate axes are not aligned with the measurement device. The affine workflow says not to fit axes by eye. Fit or receive the quadratic coefficients, symmetrize the quadratic part, find the center, diagonalize, and test residuals on points.

The lab below samples points from the ellipsoid we just constructed and checks three things:

- The original equation is nearly zero on the sampled surface.
- After translation to the center and rotation to eigen-coordinates, the equation is diagonal.
- The normalized squared axis coordinates sum to one.

This is exactly the computational value of the chapter's classification: it turns a messy-looking object into a short list of invariants and residuals.


In [ ]:
sample_angles = np.linspace(0, 2 * np.pi, 18, endpoint=False)
sample_heights = np.linspace(0.2, np.pi - 0.2, 9)
samples = []
for phi in sample_angles:
    for theta in sample_heights:
        local_point = np.array([
            computed_axes_sorted[0] * math.cos(phi) * math.sin(theta),
            computed_axes_sorted[1] * math.sin(phi) * math.sin(theta),
            computed_axes_sorted[2] * math.cos(theta),
        ])
        samples.append(local_point @ computed_directions.T + reduced["center"])
samples = np.array(samples)

residual_original = quadric_value(A3, b3, c3, samples)
centered_samples = (samples - reduced["center"]) @ computed_directions
normalized_radius = np.sum((centered_samples / computed_axes_sorted) ** 2, axis=1)

diagonal_matrix = computed_directions.T @ A3 @ computed_directions
off_diagonal_error = float(np.linalg.norm(diagonal_matrix - np.diag(np.diag(diagonal_matrix))))

lab_summary = {
    "sample_count": int(len(samples)),
    "max_original_equation_residual": float(np.max(np.abs(residual_original))),
    "max_normalized_radius_error": float(np.max(np.abs(normalized_radius - 1.0))),
    "off_diagonal_error_after_rotation": off_diagonal_error,
}
lab_summary


## 9. Pitfalls and mental checks

- A diagonal quadratic part does not imply the affine equation is reduced. Linear terms may still remain, and those terms are decisive when `A` has a nullspace.
- A center is not the same as a point on the quadric. It is a point where the affine gradient vanishes; the center may lie inside, outside, or on a singular quadric.
- A cone and a hyperboloid can share the same quadratic part. The constant term separates the singular level from nearby smooth levels.
- Real and complex classification differ. Over `C`, signs lose meaning because nonzero scalars have square roots; over `R`, signs determine empty, compact, split, and connected cases.
- Euclidean classification is finer than affine classification. An affine map can turn one ellipsoid into another; an isometry cannot change the semi-axis lengths.

## Takeaways

Affine quadrics are linear algebra with geometry attached. The symmetric matrix gives a bilinear form, the linear term determines centers or parabolic directions, and the constant chooses the level. Homogenization lets polarity compute tangent hyperplanes and conjugate directions. Euclidean structure then adds measured axes through the spectral theorem.

The habit to keep is procedural: reduce the equation, draw the normal form, then check the drawing numerically. That loop prevents the classification table from becoming a list of names detached from the equations.


In [ ]:
artifact_paths = [
    reduction_path,
    normal_forms_path,
    gallery_path,
    topology_path,
    polarity_path,
    axes_path,
]

final_sanity = {
    "translation_identity_error": translation_check["max_translation_identity_error"],
    "polarity_contact_residual_max": float(np.max(np.abs(polarity_check["contact_residuals"]))),
    "polarity_tangent_residual_max": float(np.max(np.abs(polarity_check["point_on_tangent_residuals"]))),
    "conjugacy_residual_abs": abs(polarity_check["conjugacy_residual"]),
    "axis_center_error": axis_lab_check["center_error"],
    "axis_length_error": axis_lab_check["axis_length_error"],
    "sample_equation_residual": lab_summary["max_original_equation_residual"],
    "sample_normalized_radius_error": lab_summary["max_normalized_radius_error"],
    "off_diagonal_error_after_rotation": lab_summary["off_diagonal_error_after_rotation"],
    "artifact_count": len(artifact_paths),
}

assert final_sanity["translation_identity_error"] < 1e-10
assert final_sanity["polarity_contact_residual_max"] < 1e-9
assert final_sanity["polarity_tangent_residual_max"] < 1e-9
assert final_sanity["conjugacy_residual_abs"] < 1e-12
assert final_sanity["axis_center_error"] < 1e-10
assert final_sanity["axis_length_error"] < 1e-10
assert final_sanity["sample_equation_residual"] < 1e-10
assert final_sanity["sample_normalized_radius_error"] < 1e-10
assert final_sanity["off_diagonal_error_after_rotation"] < 1e-10
assert_artifacts(artifact_paths, min_bytes=128)

sanity_path = chapter_artifact("checks", "final_sanity.json")
save_json(final_sanity, sanity_path)
assert_artifacts([sanity_path], min_bytes=128)
final_sanity
